# EDA — BRFSS 2015 (Diabetes Health Indicators)

> Chỉ để học tập, không phải chẩn đoán y khoa.

Quy tắc: mọi thống kê liên quan nhãn (tỉ lệ bệnh theo nhóm, tương quan) chỉ tính trên **tập train**; phần chất lượng dữ liệu dùng toàn bộ bảng. Notebook này không nạp tập test.

In [1]:
import json

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from diabetes_ai.data.load import load_raw
from diabetes_ai.data.schema import CLASSES, FEATURES, TARGET
from diabetes_ai.data.split import load_split
from diabetes_ai.features.feature_names import (
    AGE_LABELS, EDUCATION_LABELS, GENHLTH_LABELS, INCOME_LABELS, VI_FEATURE,
)
from diabetes_ai.paths import FIGURES_DIR, METRICS_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
df = load_raw()
split = load_split("full")
train = df.iloc[split.train].reset_index(drop=True)   # thống kê liên quan nhãn chỉ dùng train
print(len(df), len(train), split.sizes())


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name, dpi=100)
    plt.close(fig)


def rate_diabetes(col):
    """Tỉ lệ nhãn 2 (tiểu đường) theo giá trị của cột, chỉ trên train."""
    return (train[TARGET] == 2).groupby(train[col].astype(int)).mean()


253680 172502 {'train': 172502, 'val': 30442, 'test': 50736}


## 1. Chất lượng dữ liệu (toàn bộ bảng)

In [2]:
val_json = json.loads((METRICS_DIR / "data_validation.json").read_text(encoding="utf-8"))
quality = {
    "so_dong": len(df),
    "o_thieu": int(df.isna().sum().sum()),
    "dong_trung_hoan_toan": int(df.duplicated().sum()),
    "BMI_lon_hon_60": int((df["BMI"] > 60).sum()),
}
print(quality)
assert quality["dong_trung_hoan_toan"] == val_json["n_duplicates"]


{'so_dong': 253680, 'o_thieu': 0, 'dong_trung_hoan_toan': 23899, 'BMI_lon_hon_60': 805}


## 2. Phân phối nhãn (train)

In [3]:
counts = train[TARGET].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(CLASSES, counts.values)
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}".replace(",", "."), ha="center", va="bottom", fontsize=8)
ax.set_title("Phân phối nhãn 3 lớp (train)")
ax.set_ylabel("Số hồ sơ")
save(fig, "eda_01_label_distribution.png")
print((counts / counts.sum()).round(4).to_dict())


{0: 0.8424, 1: 0.0183, 2: 0.1393}


## 3. BMI theo nhóm nhãn (train)

In [4]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([train.loc[train[TARGET] == k, "BMI"] for k in range(3)], tick_labels=CLASSES, showfliers=False)
ax.set_title("BMI theo nhóm nhãn (train, ẩn điểm ngoại lai)")
ax.set_ylabel("BMI")
save(fig, "eda_02_bmi_by_class.png")
print(train.groupby(TARGET)["BMI"].median().round(1).to_dict())


{0: 27.0, 1: 30.0, 2: 31.0}

## 4. Tỉ lệ tiểu đường theo nhóm tuổi (train)

In [5]:
r = rate_diabetes("Age")
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar([AGE_LABELS[k] for k in r.index], r.values * 100)
ax.set_title("Tỉ lệ tiểu đường (%) theo nhóm tuổi (train)")
ax.set_ylabel("%")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
save(fig, "eda_03_prevalence_by_age.png")


## 5. Tỉ lệ tiểu đường theo thu nhập và học vấn (train)

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, col, labels, title in [
    (axes[0], "Income", INCOME_LABELS, "Thu nhập"),
    (axes[1], "Education", EDUCATION_LABELS, "Học vấn"),
]:
    r = rate_diabetes(col)
    ax.bar([str(labels[k]) for k in r.index], r.values * 100)
    ax.set_title(f"Tỉ lệ tiểu đường (%) theo {title}")
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right", fontsize=7)
save(fig, "eda_04_prevalence_by_income_education.png")


## 6. Tỉ lệ tiểu đường theo tự đánh giá sức khỏe (train)

In [7]:
r = rate_diabetes("GenHlth")
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar([str(GENHLTH_LABELS[k]) for k in r.index], r.values * 100)
ax.set_title("Tỉ lệ tiểu đường (%) theo GenHlth (train)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
save(fig, "eda_05_prevalence_by_genhlth.png")


## 7. Yếu tố nhị phân: chênh lệch tỉ lệ tiểu đường khi có / không có yếu tố (train)

In [8]:
bin_cols = [c for c in FEATURES if train[c].nunique() == 2]
is_dia = train[TARGET] == 2
gap = pd.Series({c: 100 * (is_dia[train[c] == 1].mean() - is_dia[train[c] == 0].mean()) for c in bin_cols}).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([VI_FEATURE[c] for c in gap.index], gap.values)
ax.set_title("Chênh lệch tỉ lệ tiểu đường (điểm %): có − không (train)")
save(fig, "eda_06_binary_factor_gap.png")


## 8. Ma trận tương quan (train)

In [9]:
corr = train[FEATURES + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns, fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Tương quan Pearson (train)")
save(fig, "eda_07_correlation.png")


## 9. Kết luận (số liệu in từ kết quả chạy)

In [10]:
top = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(5).index
print("Tỉ lệ tiểu đường nền (train): %.2f%%" % (100 * (train[TARGET] == 2).mean()))
print("5 đặc trưng tương quan mạnh nhất với nhãn:", {k: round(float(corr.loc[k, TARGET]), 3) for k in top})
print("Lưu ý: tương quan không phải nhân quả; dữ liệu khảo sát tự khai ở Hoa Kỳ năm 2015.")


Tỉ lệ tiểu đường nền (train): 13.93%
5 đặc trưng tương quan mạnh nhất với nhãn: {'GenHlth': 0.305, 'HighBP': 0.273, 'BMI': 0.226, 'DiffWalk': 0.226, 'HighChol': 0.209}
Lưu ý: tương quan không phải nhân quả; dữ liệu khảo sát tự khai ở Hoa Kỳ năm 2015.
